In [1]:
!pip install pyspark

In [3]:
spark = (SparkSession.builder.appName("WarehouseInsights").master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())


In [4]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DateType, IntegerType, StringType, StructField, StructType

In [5]:
from google.colab import files
uploaded = files.upload()

Saving stock_movements_clean.csv to stock_movements_clean.csv
Uploaded: ['stock_movements_clean.csv']


In [6]:
from google.colab import files
uploaded = files.upload()

Saving warehouses.csv to warehouses.csv


In [7]:
from google.colab import files
uploaded = files.upload()

Saving products.csv to products.csv


In [8]:
movement_schema = StructType([
    StructField("movement_id",   IntegerType(), False),
    StructField("product_id",    IntegerType(), False),
    StructField("warehouse_id",  IntegerType(), False),
    StructField("movement_type", StringType(),  True),
    StructField("quantity",      IntegerType(), False),
    StructField("movement_date", DateType(),    True),
    StructField("reference_no",  StringType(),  True),
    StructField("reason",        StringType(),  True),
])

movements  = spark.read.csv("stock_movements_clean.csv", header=True, schema=movement_schema)
products   = spark.read.csv("products.csv",   header=True, inferSchema=True)
warehouses = spark.read.csv("warehouses.csv", header=True, inferSchema=True)

print("movements:", movements.count(), "| products:", products.count(), "| warehouses:", warehouses.count())
movements.show(5)

movements: 3813 | products: 48 | warehouses: 5
+-----------+----------+------------+-------------+--------+-------------+------------+-------------+
|movement_id|product_id|warehouse_id|movement_type|quantity|movement_date|reference_no|reason       |
+-----------+----------+------------+-------------+--------+-------------+------------+-------------+
|1          |1         |1           |IN           |93      |2026-01-13   |OPEN-1-1    |Opening stock|
|2          |1         |2           |IN           |141     |2026-01-13   |OPEN-1-2    |Opening stock|
|3          |1         |3           |IN           |130     |2026-01-13   |OPEN-1-3    |Opening stock|
|4          |1         |4           |IN           |125     |2026-01-13   |OPEN-1-4    |Opening stock|
|5          |1         |5           |IN           |137     |2026-01-13   |OPEN-1-5    |Opening stock|
+-----------+----------+------------+-------------+--------+-------------+------------+-------------+
only showing top 5 rows


In [10]:
stock = movements.groupBy("product_id", "warehouse_id").agg(
    F.sum("quantity").alias("quantity_on_hand"),
    F.max("movement_date").alias("last_movement_date"))

detail = (stock
          .join(F.broadcast(products), "product_id")
          .join(F.broadcast(warehouses.select("warehouse_id", "warehouse_name", "city")), "warehouse_id")
          .withColumn("stock_status",
                      F.when(F.col("quantity_on_hand") < F.col("reorder_level"), "LOW_STOCK")
                       .when(F.col("quantity_on_hand") > F.col("max_stock_level"), "OVERSTOCK")
                       .otherwise("OK"))
          .withColumn("stock_value", F.col("quantity_on_hand") * F.col("unit_price")))
detail.cache()

print("Product-warehouse rows:", detail.count())
detail.select("warehouse_name", "sku", "product_name", "quantity_on_hand",
              "reorder_level", "max_stock_level", "stock_status").show()

Product-warehouse rows: 240
+--------------------+--------+--------------------+----------------+-------------+---------------+------------+
|      warehouse_name|     sku|        product_name|quantity_on_hand|reorder_level|max_stock_level|stock_status|
+--------------------+--------+--------------------+----------------+-------------+---------------+------------+
|Bengaluru Tech Pa...|ELE-0004| Power Bank 10000mAh|              48|           25|            125|          OK|
| Hyderabad South Hub|ELE-0006|        Webcam 1080p|             451|          100|            500|          OK|
|Bengaluru Tech Pa...|ELE-0007|        Laptop Stand|              19|          105|            525|   LOW_STOCK|
|      Delhi North DC|STA-0012|             Stapler|              22|           75|            450|   LOW_STOCK|
|Bengaluru Tech Pa...|STA-0012|             Stapler|             651|           75|            450|   OVERSTOCK|
|      Delhi North DC|STA-0015|Whiteboard Marker...|             256

In [11]:
total_per_warehouse = (movements.groupBy("warehouse_id")
                       .agg(F.sum("quantity").alias("total_units"))
                       .join(warehouses.select("warehouse_id", "warehouse_name"), "warehouse_id")
                       .orderBy("warehouse_id"))
total_per_warehouse.show()

+------------+-----------+--------------------+
|warehouse_id|total_units|      warehouse_name|
+------------+-----------+--------------------+
|           1|       7101|  Chennai Central DC|
|           2|      14285|     Mumbai West Hub|
|           3|      10755|      Delhi North DC|
|           4|      14470|Bengaluru Tech Pa...|
|           5|      10591| Hyderabad South Hub|
+------------+-----------+--------------------+



In [12]:
THRESHOLD_PCT = 25.0
warehouse_status = (detail.groupBy("warehouse_id", "warehouse_name", "city").agg(
        F.sum("quantity_on_hand").alias("total_units"),
        F.round(F.sum("stock_value"), 2).alias("total_stock_value"),
        F.count("*").alias("product_lines"),
        F.sum(F.when(F.col("stock_status") == "LOW_STOCK", 1).otherwise(0)).alias("low_stock_items"),
        F.sum(F.when(F.col("stock_status") == "OVERSTOCK", 1).otherwise(0)).alias("overstock_items"))
    .withColumn("low_stock_pct",  F.round(100 * F.col("low_stock_items")  / F.col("product_lines"), 1))
    .withColumn("overstock_pct",  F.round(100 * F.col("overstock_items") / F.col("product_lines"), 1))
    .withColumn("warehouse_status",
                F.when((F.col("low_stock_pct") >= THRESHOLD_PCT) &
                       (F.col("low_stock_pct") >= F.col("overstock_pct")), "UNDERSTOCKED")
                 .when(F.col("overstock_pct") >= THRESHOLD_PCT, "OVERSTOCKED")
                 .otherwise("BALANCED"))
    .orderBy("warehouse_id"))

warehouse_status.show()

+------------+--------------------+---------+-----------+-----------------+-------------+---------------+---------------+-------------+-------------+----------------+
|warehouse_id|      warehouse_name|     city|total_units|total_stock_value|product_lines|low_stock_items|overstock_items|low_stock_pct|overstock_pct|warehouse_status|
+------------+--------------------+---------+-----------+-----------------+-------------+---------------+---------------+-------------+-------------+----------------+
|           1|  Chennai Central DC|  Chennai|       7101|          5091092|           48|             21|              4|         43.8|          8.3|    UNDERSTOCKED|
|           2|     Mumbai West Hub|   Mumbai|      14285|          9199598|           48|              4|             17|          8.3|         35.4|     OVERSTOCKED|
|           3|      Delhi North DC|    Delhi|      10755|          7459819|           48|              6|              7|         12.5|         14.6|        BALANCED

In [13]:
THRESHOLD_PCT = 25.0   # flag a warehouse when 25% or more of its product lines are low/over

warehouse_status = (detail.groupBy("warehouse_id", "warehouse_name", "city").agg(
        F.sum("quantity_on_hand").alias("total_units"),
        F.round(F.sum("stock_value"), 2).alias("total_stock_value"),
        F.count("*").alias("product_lines"),
        F.sum(F.when(F.col("stock_status") == "LOW_STOCK", 1).otherwise(0)).alias("low_stock_items"),
        F.sum(F.when(F.col("stock_status") == "OVERSTOCK", 1).otherwise(0)).alias("overstock_items"))
    .withColumn("low_stock_pct",  F.round(100 * F.col("low_stock_items")  / F.col("product_lines"), 1))
    .withColumn("overstock_pct",  F.round(100 * F.col("overstock_items") / F.col("product_lines"), 1))
    .withColumn("warehouse_status",
                F.when((F.col("low_stock_pct") >= THRESHOLD_PCT) &
                       (F.col("low_stock_pct") >= F.col("overstock_pct")), "UNDERSTOCKED")
                 .when(F.col("overstock_pct") >= THRESHOLD_PCT, "OVERSTOCKED")
                 .otherwise("BALANCED"))
    .orderBy("warehouse_id"))

warehouse_status.show()

+------------+--------------------+---------+-----------+-----------------+-------------+---------------+---------------+-------------+-------------+----------------+
|warehouse_id|      warehouse_name|     city|total_units|total_stock_value|product_lines|low_stock_items|overstock_items|low_stock_pct|overstock_pct|warehouse_status|
+------------+--------------------+---------+-----------+-----------------+-------------+---------------+---------------+-------------+-------------+----------------+
|           1|  Chennai Central DC|  Chennai|       7101|          5091092|           48|             21|              4|         43.8|          8.3|    UNDERSTOCKED|
|           2|     Mumbai West Hub|   Mumbai|      14285|          9199598|           48|              4|             17|          8.3|         35.4|     OVERSTOCKED|
|           3|      Delhi North DC|    Delhi|      10755|          7459819|           48|              6|              7|         12.5|         14.6|        BALANCED

In [14]:
exceptions = (detail.filter(F.col("stock_status") != "OK")
              .select("warehouse_name", "sku", "product_name", "category",
                      "quantity_on_hand", "reorder_level", "max_stock_level", "stock_status")
              .orderBy("warehouse_name", "stock_status", "sku"))

print("Items needing attention:", exceptions.count())
print("\nUNDERSTOCKED items (below reorder level):")
exceptions.filter(F.col("stock_status") == "LOW_STOCK").show(10)
print("OVERSTOCKED items (above maximum level):")
exceptions.filter(F.col("stock_status") == "OVERSTOCK").show(10)

Items needing attention: 87

UNDERSTOCKED items (below reorder level):
+--------------------+--------+--------------------+----------------+----------------+-------------+---------------+------------+
|      warehouse_name|     sku|        product_name|        category|quantity_on_hand|reorder_level|max_stock_level|stock_status|
+--------------------+--------+--------------------+----------------+----------------+-------------+---------------+------------+
|Bengaluru Tech Pa...|ELE-0007|        Laptop Stand|     Electronics|              19|          105|            525|   LOW_STOCK|
|Bengaluru Tech Pa...|HOK-0021|Dinner Plate Set 6pc|  Home & Kitchen|              25|          115|            690|   LOW_STOCK|
|Bengaluru Tech Pa...|TOH-0027|   Measuring Tape 5m|Tools & Hardware|              43|           75|            450|   LOW_STOCK|
|  Chennai Central DC|ELE-0003|   Bluetooth Speaker|     Electronics|               2|           25|            150|   LOW_STOCK|
|  Chennai Central 

In [16]:
import os
import pandas as pd

os.makedirs("output", exist_ok=True)
warehouse_status.toPandas().to_csv("output/warehouse_stock_status.csv", index=False)
exceptions.toPandas().to_csv("output/warehouse_item_exceptions.csv", index=False)

pd.read_csv("output/warehouse_stock_status.csv")

,warehouse_id,warehouse_name,city,total_units,total_stock_value,product_lines,low_stock_items,overstock_items,low_stock_pct,overstock_pct,warehouse_status
0,1,Chennai Central DC,Chennai,7101,5091092,48,21,4,43.8,8.3,UNDERSTOCKED
1,2,Mumbai West Hub,Mumbai,14285,9199598,48,4,17,8.3,35.4,OVERSTOCKED
2,3,Delhi North DC,Delhi,10755,7459819,48,6,7,12.5,14.6,BALANCED
3,4,Bengaluru Tech Park WH,Bengaluru,14470,8605403,48,3,17,6.3,35.4,OVERSTOCKED
4,5,Hyderabad South Hub,Hyderabad,10591,7384188,48,2,6,4.2,12.5,BALANCED


In [17]:
import shutil
shutil.make_archive("week3_output", "zip", "output")
files.download("week3_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>